In [ ]:
from flask import Flask, request, jsonify, render_template
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib
from datetime import datetime, timedelta

app = Flask(__name__)

# ---------------------------
# 1. Define your model class
#    (must match training)
# ---------------------------

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, features)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # use last time-step
        return out  # (batch, output_size)


# ---------------------------
# 2. Load model + artifacts
# ---------------------------

device = torch.device("cpu")

# Sequence length and feature count saved from training
SEQ_LEN = int(np.load("seq_len.npy")[0])
N_FEATURES = int(np.load("n_features.npy")[0])

# ⚠️ These MUST match your training hyperparameters
HIDDEN_SIZE = 64       # <-- set this to the same value as in training
NUM_LAYERS = 1         # <-- important: 1 layer, as indicated by your error
OUTPUT_SIZE = N_FEATURES  # model outputs full feature vector

# Init model and load weights
model_all = LSTMModel(
    input_size=N_FEATURES,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    output_size=OUTPUT_SIZE
).to(device)

model_all.load_state_dict(torch.load("model_all.pth", map_location=device))
model_all.eval()

# Load scaler and historical data
scaler = joblib.load("scaler.pkl")
df_hist = pd.read_csv("historical_data.csv")

# Ensure Date column is datetime and sorted
df_hist["Date"] = pd.to_datetime(df_hist["Date"])
df_hist = df_hist.sort_values("Date")


# ---------------------------
# 3. Helper: build last sequence
# ---------------------------

def _build_last_sequence():
    """
    Builds the last input sequence to the model:
    shape: (1, SEQ_LEN, N_FEATURES)

    We try to use the same feature order used by the scaler.
    """
    # Prefer the column order used when fitting the scaler (if available)
    if hasattr(scaler, "feature_names_in_"):
        feature_cols = list(scaler.feature_names_in_)
    else:
        # Fallback: all columns except Date
        feature_cols = [c for c in df_hist.columns if c != "Date"]

    data_values = df_hist[feature_cols].values
    data_scaled = scaler.transform(data_values)

    if len(data_scaled) < SEQ_LEN:
        raise ValueError(f"Not enough rows to build a sequence of length {SEQ_LEN}.")

    last_seq = data_scaled[-SEQ_LEN:]               # (seq_len, features)
    last_seq = np.expand_dims(last_seq, axis=0)     # (1, seq_len, features)
    last_seq = torch.from_numpy(last_seq).float().to(device)

    return last_seq, feature_cols


# ---------------------------
# 4. Forecast helpers
# ---------------------------

def forecast_next_n_days(n: int):
    """
    Returns a list with n days of predictions,
    starting from the day AFTER the last date in df_hist.

    Each element: {date, open, high, low, close}
    """
    last_sequence, feature_cols = _build_last_sequence()
    last_date = df_hist["Date"].iloc[-1].date()

    preds = []
    seq = last_sequence.clone()

    with torch.no_grad():
        for i in range(n):
            # Forward pass
            pred = model_all(seq)        # (1, N_FEATURES)
            pred = pred.unsqueeze(1)     # (1, 1, N_FEATURES)

            # Append prediction to sequence (autoregressive)
            seq = torch.cat((seq, pred), dim=1)  # (1, seq_len+1, N_FEATURES)
            seq = seq[:, 1:, :]                  # keep last seq_len

            # Convert prediction back to original scale
            pred_np = pred.squeeze(0).squeeze(0).cpu().numpy()  # (N_FEATURES,)
            pred_inv = scaler.inverse_transform(pred_np.reshape(1, -1))[0]

            pred_date = last_date + timedelta(days=i + 1)

            # Assuming first 4 features are Open, High, Low, Close
            preds.append({
                "date": pred_date.strftime("%Y-%m-%d"),
                "open": float(pred_inv[0]),
                "high": float(pred_inv[1]),
                "low": float(pred_inv[2]),
                "close": float(pred_inv[3]),
            })

    return preds


def forecast_until_date(target_date_str: str):
    """
    Forecast for a specific future date (YYYY-MM-DD).
    """
    target_date = datetime.strptime(target_date_str, "%Y-%m-%d").date()
    last_date = df_hist["Date"].iloc[-1].date()

    if target_date <= last_date:
        return {"error": f"Target date must be AFTER last historical date {last_date}."}

    days_ahead = (target_date - last_date).days

    last_sequence, feature_cols = _build_last_sequence()
    seq = last_sequence.clone()

    with torch.no_grad():
        for _ in range(days_ahead):
            pred = model_all(seq)        # (1, N_FEATURES)
            pred = pred.unsqueeze(1)     # (1, 1, N_FEATURES)
            seq = torch.cat((seq, pred), dim=1)
            seq = seq[:, 1:, :]

    # Final sequence's last time-step = target date prediction
    final_seq = seq.squeeze(0).cpu().numpy()        # (seq_len, N_FEATURES)
    pred_scaled = final_seq[-1, :]                  # (N_FEATURES,)
    pred_inv = scaler.inverse_transform(pred_scaled.reshape(1, -1))[0]

    return {
        "date": target_date_str,
        "open": float(pred_inv[0]),
        "high": float(pred_inv[1]),
        "low": float(pred_inv[2]),
        "close": float(pred_inv[3]),
    }


# ---------------------------
# 5. Routes
# ---------------------------

@app.route("/", methods=["GET", "POST"])
def home():
    # Always show 1-week forecast on home
    week_forecast = forecast_next_n_days(7)

    prediction_result = None
    error_message = None

    if request.method == "POST":
        target_date = request.form.get("target_date")
        if not target_date:
            error_message = "Please select a date."
        else:
            try:
                res = forecast_until_date(target_date)
                if "error" in res:
                    error_message = res["error"]
                else:
                    prediction_result = res
            except Exception as e:
                error_message = str(e)

    return render_template(
        "index.html",
        week_forecast=week_forecast,
        prediction_result=prediction_result,
        error_message=error_message
    )


# Optional JSON API endpoint
@app.route("/api/forecast", methods=["GET"])
def api_forecast():
    target_date = request.args.get("date")
    if not target_date:
        return jsonify({"error": "Please pass ?date=YYYY-MM-DD"}), 400

    res = forecast_until_date(target_date)
    if "error" in res:
        return jsonify(res), 400
    return jsonify(res), 200


if __name__ == "__main__":
    app.run(debug=False,port=5001)


 * Serving Flask app '__main__'
 * Debug mode: off


C:\Users\User\AppData\Local\Temp\ipykernel_24252\3163392218.py:65: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_hist["Date"] = pd.to_datetime(df_hist["Date"])
 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
[2025-11-17 17:44:31,022] ERROR in app: Exception on / [GET]
Traceback (most recent call last):
  File "C:\Users\User\anaconda3\envs\stock_price_prediction\lib\site-packages\flask\app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\User\anaconda3\envs\stock_price_prediction\lib\site-packages\flask\app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\User\anaconda3\envs\stock_price_prediction\lib\site-packages\fla